# 03. Carry gap 평균회귀
가설4 검정

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy import stats

from common.transforms import funding_to_daily_annualized, build_spread, log_return
from common.stats import ols_hac
from common.visualization import plot_timeseries

In [ ]:
RAW = Path('../../data/raw')
PROCESSED = Path('../../data/processed')

# funding 파일은 auto 수집 결과에 따라 binance_... 또는 bybit_... 일 수 있음
eth_f_path = next((p for p in [RAW/'binance_ethusdt_funding.csv', RAW/'bybit_ethusdt_funding.csv'] if p.exists()), None)
btc_f_path = next((p for p in [RAW/'binance_btcusdt_funding.csv', RAW/'bybit_btcusdt_funding.csv'] if p.exists()), None)

assert eth_f_path and btc_f_path, 'funding 파일이 없습니다. scripts/collect_binance_data.py 먼저 실행하세요.'

eth_f = pd.read_csv(eth_f_path)
btc_f = pd.read_csv(btc_f_path)
eth_p = pd.read_csv(RAW/'binance_ethusdt_1d.csv')
btc_p = pd.read_csv(RAW/'binance_btcusdt_1d.csv')

# staking yield 패널(권장): scripts/build_eth_yield_panel.py 결과물
st_path = PROCESSED/'eth_yield_panel.csv'
if st_path.exists():
    st = pd.read_csv(st_path)
else:
    # fallback: 직접 raw 파일 사용
    st_path = RAW/'eth_staking_yield_daily.csv'
    assert st_path.exists(), 'eth_yield_panel.csv 또는 eth_staking_yield_daily.csv 가 필요합니다.'
    st = pd.read_csv(st_path)

In [ ]:
# 1) funding -> daily annualized
eth_fd = funding_to_daily_annualized(eth_f, time_col='fundingTime', rate_col='fundingRate')
btc_fd = funding_to_daily_annualized(btc_f, time_col='fundingTime', rate_col='fundingRate')

# 2) spread 생성
spread = build_spread(eth_fd.rename(columns={'funding_ann':'funding_ann'}), btc_fd.rename(columns={'funding_ann':'funding_ann'}), value_col='funding_ann', out_col='spread')

# 3) 가격 수익률 생성
for px in (eth_p, btc_p):
    px['date'] = pd.to_datetime(px['date'])
eth_p['ret_eth'] = log_return(eth_p['close'])
btc_p['ret_btc'] = log_return(btc_p['close'])
ret = eth_p[['date','ret_eth']].merge(btc_p[['date','ret_btc']], on='date', how='inner')
ret['ret_eth_btc'] = ret['ret_eth'] - ret['ret_btc']

# 단순 RV proxy (절대수익률)
ret['rv_eth_btc'] = (ret['ret_eth'].abs() - ret['ret_btc'].abs())

# staking
st['date'] = pd.to_datetime(st['date'])
st['stake_yield'] = pd.to_numeric(st['stake_yield'], errors='coerce')
# stake_yield가 % 값이면 100으로 나누기 (예: 3.5 -> 0.035)
if st['stake_yield'].dropna().median() > 1:
    st['stake_yield'] = st['stake_yield'] / 100.0

spread['date'] = pd.to_datetime(spread['date'])
df = spread[['date','spread']].merge(ret[['date','ret_eth_btc','rv_eth_btc']], on='date', how='inner').merge(st[['date','stake_yield']], on='date', how='inner').dropna().sort_values('date').reset_index(drop=True)

df.head(), df.describe().T[['mean','std','min','max']]

In [ ]:
# carry gap 생성
df_mr = df.copy()
df_mr['carry_gap'] = df_mr['spread'] + df_mr['stake_yield']
df_mr['d_carry_gap_f1'] = df_mr['carry_gap'].shift(-1) - df_mr['carry_gap']
df_mr = df_mr.dropna().reset_index(drop=True)
df_mr[['date','carry_gap','d_carry_gap_f1']].head()

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(11,7),sharex=True)
plot_timeseries(df_mr, 'date', 'carry_gap', 'Carry Gap', ax=axes[0])
plot_timeseries(df_mr, 'date', 'd_carry_gap_f1', 'Delta Carry Gap (t+1)', ax=axes[1])
plt.tight_layout()

In [ ]:
# 평균회귀 회귀식: ΔCarryGap_{t+1} = a + rho * CarryGap_t + e
res_mr = ols_hac(df_mr['d_carry_gap_f1'], df_mr[['carry_gap']], maxlags=5)
print(res_mr.summary())

In [ ]:
# winsorize(1%) 강건성
q1, q99 = df_mr['carry_gap'].quantile([0.01, 0.99])
rob = df_mr.copy()
rob['carry_gap_w'] = rob['carry_gap'].clip(q1, q99)
rob['d_carry_gap_f1_w'] = rob['carry_gap_w'].shift(-1) - rob['carry_gap_w']
rob = rob.dropna()
res_w = ols_hac(rob['d_carry_gap_f1_w'], rob[['carry_gap_w']], maxlags=5)
print(res_w.summary())